In [13]:
import numpy as np
import pandas as pd
import os
import tensorflow as tf
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import timm

from collections import Counter
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from keras.models import Sequential
from keras.layers import LSTM, Dense, Conv2D, MaxPooling2D, Flatten, Dropout
from keras.utils import to_categorical
from imblearn.over_sampling import SMOTE
#from pyts.image import GramianAngularField, MarkovTransitionField
from io import BytesIO
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [14]:
BASE = '/home/HardDisk/Satang/thesis_proj'

In [15]:
DATAPATH = 'original'

In [16]:
FOLDER = "win7-120gb-hdd"

In [17]:
os.chdir(f'{BASE}/{DATAPATH}')
folders = sorted(os.listdir())
print(folders)

['win7-120gb-hdd', 'win7-120gb-ssd', 'win7-250gb-hdd', 'win7-250gb-ssd']


In [18]:
os.chdir(f'{BASE}/{DATAPATH}/{FOLDER}')
labels = sorted(os.listdir())
print(labels)

['AESCrypt', 'Cerber', 'Darkside', 'Excel', 'Firefox', 'GandCrab4', 'Ryuk', 'SDelete', 'Sodinokibi', 'TeslaCrypt', 'WannaCry', 'Zip']


In [19]:
benign = ['AESCrypt', 'Zip', 'SDelete', 'Excel', 'Firefox']
ransomware = ['TeslaCrypt', 'Cerber', 'WannaCry', 'GandCrab4', 'Ryuk', 'Sodinokibi', 'Darkside']

In [20]:
import numpy as np
import os

def save_numpy_array(array, label, base_dir, file_format='npy'):
    """
    Save a Numpy array to a file in a class-named folder, appending a dynamic suffix to the label based on the number of existing files.

    Parameters:
    - array (np.ndarray): The Numpy array to save.
    - label (str): The base label to use as the filename and folder name for the class.
    - base_dir (str): The base directory where the class folder will be created.
    - file_format (str): The format to save the file in ('npy' or 'csv'). Default is 'npy'.
    
    Returns:
    - str: The full path to the saved file.
    """
    # Create class folder within the base directory
    class_dir = os.path.join(base_dir, label)
    os.makedirs(class_dir, exist_ok=True)  # Create the folder if it doesn't exist

    # Count existing files in the class folder
    existing_files = os.listdir(class_dir)
    file_count = sum(1 for file in existing_files if file.endswith(f".{file_format}"))
    
    # Generate suffix based on the file count
    suffix = f"_{file_count + 1}"  # Start from _1 if no files exist

    # Define the full file path with the dynamic suffix
    file_name = f"{label}{suffix}.{file_format}"
    file_path = os.path.join(class_dir, file_name)

    # Save the file in the specified format
    if file_format == 'npy':
        np.save(file_path, array)
    elif file_format == 'csv':
        np.savetxt(file_path, array, delimiter=",", comments="")
    else:
        raise ValueError("Unsupported file format. Use 'npy' or 'csv'.")

    return file_path

In [21]:

# # Function to calculate the sum of corresponding values in column 2 based on groups in column 1
# def count_operations_and_sum(column_1, column_2):
#     # Get start and end dynamically using np.min and np.max
#     start = np.min(column_1)
#     end = np.max(column_1)
#     unique_values = np.unique(column_1)
#     operations = []
#     sum_dict = {}  # Dictionary to store sums of values for each operation
#     count_dict = {}  # Dictionary to store count of times each number appears for each operation
    
#     # Identify the valid steps between start and end
#     i = np.where(unique_values == start)[0][0]  # Find the index of the start value in unique_values
#     while i < len(unique_values) - 2 and unique_values[i+2] <= end:
#         selected_values = unique_values[i:i+3]
        
#         # Check if all three numbers exist in column_1
#         if all(value in column_1 for value in selected_values):
#             operations.append(selected_values)
            
#             # Get the indices of the selected values in column_1
#             indices = [np.where(column_1 == value)[0] for value in selected_values]
            
#             # Sum the corresponding values in column_2
#             sum_value = sum([column_2[index].sum() for index in indices])
#             sum_dict[tuple(selected_values)] = sum_value
            
#             # Count how many times each number appears
#             count_value = sum([len(np.where(column_1 == value)[0]) for value in selected_values])
#             count_dict[tuple(selected_values)] = count_value
        
#         # Skip over missing numbers and move to the next distinct number
#         i += 1
    
#     return operations, sum_dict, count_dict

# # Calculate the number of operations, sums, and counts
# operations, sum_dict, count_dict = count_operations_and_sum(column_1, column_2)

# # Output the operations
# print("Operations to perform:", operations)
# print("Number of operations:", len(operations))

# # Output the sum and the count of each operation
# print("\nSum of each operation (based on column 2 values) and count of times each number appears:")
# for operation in operations:
#     operation_sum = sum_dict[tuple(operation)]
#     operation_count = count_dict[tuple(operation)]
#     print(f"{operation}: Sum = {operation_sum}, Count = {operation_count}")

In [22]:
import numpy as np

df_r = pd.read_csv("/home/HardDisk/Satang/thesis_proj/dataset/dataset/original/win7-120gb-hdd/AESCrypt/AESCrypt-20200427_16-23-28/ata_write.csv",header=None)
df_r = np.array(df_r)

# Get unique values in order of appearance and their counts
column_1 = df_r[:,0].astype(int)
column_2 = df_r[:,3]
# Get unique values in column 1 and their counts
unique_values = np.unique(column_1)

def count_operations_and_sum(column_1, column_2, window_size=15):
    # Step 1: Count unique values and occurrences
    unique_values, counts = np.unique(column_1, return_counts=True)
    unique_counts_array = np.array(counts)  # Store unique counts in an array

    # Step 2: Compute sums and counts for `column_2`
    sum_dict = {}  # Store sum of column_2 values for each operation
    count_dict = {}  # Store count of occurrences for each operation
    operations = []  # Store the sequences (as NumPy arrays)
    sliding_sums = []  # Store sum of unique counts in each sliding window

    # Identify the valid steps between start and end
    for i in range(len(unique_values) - (window_size-1)):  # Move one step at a time
        operation = unique_values[i:i+window_size]  # Extract 3 consecutive values
        operation_key = str(operation)  # Convert to string for dictionary key storage

        if operation_key not in sum_dict:
            mask = np.isin(column_1, operation)  # Find rows where column_1 matches the operation
            sum_dict[operation_key] = np.nansum(column_2[mask])  # Sum corresponding column_2 values
            count_dict[operation_key] = np.sum(mask)  # Count occurrences
            operations.append(operation)  # Append as a NumPy array



    return operations, sum_dict, count_dict, unique_counts_array



operations, sum_dict, count_dict, unique_counts = count_operations_and_sum(column_1, column_2)

print("Operations:", len(operations))
print("Sum Dict:", sum_dict)
print("Count Dict:", count_dict)
print("\nUnique Counts Array:", unique_counts)

# operation_r = str(operations[1])
# print(sum_dict[operation_r])

Operations: 87
Sum Dict: {'[1587971914 1587971915 1587971916 1587971917 1587971918 1587971919\n 1587971920 1587971921 1587971922 1587971923 1587971924 1587971925\n 1587971926 1587971927 1587971928]': 51704320.0, '[1587971915 1587971916 1587971917 1587971918 1587971919 1587971920\n 1587971921 1587971922 1587971923 1587971924 1587971925 1587971926\n 1587971927 1587971928 1587971929]': 73908736.0, '[1587971916 1587971917 1587971918 1587971919 1587971920 1587971921\n 1587971922 1587971923 1587971924 1587971925 1587971926 1587971927\n 1587971928 1587971929 1587971930]': 73875968.0, '[1587971917 1587971918 1587971919 1587971920 1587971921 1587971922\n 1587971923 1587971924 1587971925 1587971926 1587971927 1587971928\n 1587971929 1587971930 1587971931]': 96264704.0, '[1587971918 1587971919 1587971920 1587971921 1587971922 1587971923\n 1587971924 1587971925 1587971926 1587971927 1587971928 1587971929\n 1587971930 1587971931 1587971932]': 96203264.0, '[1587971919 1587971920 1587971921 158797192

In [23]:
# import pandas as pd

# # Load your dataset
# unique_values, counts = np.unique(df_r[:,0], return_counts=True)

# # Display the results
# for value, count in zip(unique_values, counts):
#     print(f"Value {value}: {count} times")


## Sliding windows

In [24]:
SEED = 42
output_dir = "/home/HardDisk/Satang/thesis_proj/New_45/10/raw_data_8"
np.random.seed(SEED)
window_size = 10
for folder in folders:
    for label in labels:
        os.chdir(f'{BASE}/{DATAPATH}/{folder}/{label}')
        dirs = sorted(os.listdir())
        dirs = np.array(dirs)
        # Shuffle directory
        np.random.seed(SEED)
        np.random.shuffle(dirs)

        for dir_idx in range(len(dirs)):
            print(dirs[dir_idx])
            os.chdir(f'{BASE}/{DATAPATH}/{folder}/{label}/{dirs[dir_idx]}')
            files = sorted(os.listdir())
            tmp = []
            tmp_train = []
            
            df_r = pd.read_csv(f'{BASE}/{DATAPATH}/{folder}/{label}/{dirs[dir_idx]}/{files[0]}', header=None)
            df_w = pd.read_csv(f'{BASE}/{DATAPATH}/{folder}/{label}/{dirs[dir_idx]}/{files[1]}', header=None)
            # df_r = np.array(df_r)
            # df_w = np.array(df_w)

            # Given column 1 (numbers) and column 2 (values to sum)
            column_r = df_r.iloc[:, 0]
            column_w = df_w.iloc[:, 0]

            values_r = df_r.iloc[:, 3]
            values_w = df_w.iloc[:, 3]

            # Get unique operations for read and write
            operations_r, sum_dict_r, count_dict_r, unique_count_r = count_operations_and_sum(column_r, values_r)
            operations_w, sum_dict_w, count_dict_w, unique_count_w = count_operations_and_sum(column_w, values_w)
            # Ensure operations exist before processing

            # Ensure operations exist before processing
            step = min(len(operations_r), len(operations_w))
            i_r = 0
            i_w = 0
            for k in range(step):
                operation_r = str(operations_r[k])  # Convert NumPy array to string key
                operation_w = str(operations_w[k])  # Convert NumPy array to string key

                # Get operation sums and counts separately
                operation_sum_r = sum_dict_r[operation_r]
                operation_sum_w = sum_dict_w[operation_w]

                count_r = count_dict_r[operation_r]
                count_w = count_dict_w[operation_w]

                # print(count_r)
                # print(count_w)

                unique_r = unique_count_r[k:k+window_size]
                unique_w = unique_count_w[k:k+window_size]
                # print(k)
                # print(unique_r)
                # print(unique_w)
                # Average read/write throughput [byte/s]
                T_read = operation_sum_r / window_size
                T_write = operation_sum_w / window_size

                # print(T_read)
                # print(T_write)
                
                # Variance of logical block addresses (read)
                filtered_read = df_r.iloc[i_r:i_r + np.sum(unique_r), 2]
                filtered_read = filtered_read[~np.isnan(filtered_read)]  # Remove NaN
                # print(filtered_read.shape)
                V_read_mean = np.mean(filtered_read)
                # print(V_read_mean)
                V_read = (1 / (count_r - 1)) * np.sum((filtered_read - V_read_mean) ** 2)

                # Variance of logical block addresses (write)
                filtered_write = df_w.iloc[i_w:i_w + np.sum(unique_w), 2]
                filtered_write = filtered_write[~np.isnan(filtered_write)]  # Remove NaN
                V_write_mean = np.mean(filtered_write)
                V_write = (1 / (count_w - 1)) * np.sum((filtered_write - V_write_mean) ** 2)

                # Average normalized Shannon entropy (write)
                filtered_entropy = df_w.iloc[i_w:i_w + np.sum(unique_w), 4]
                filtered_entropy = filtered_entropy[~np.isnan(filtered_entropy)]  # Remove NaN
                entropy_mean = np.mean(filtered_entropy)
                num_row_ent = filtered_entropy.shape[0]
                if num_row_ent > 0:
                    H_write = (1 / count_w) * np.sum(df_w.iloc[i_w:i_w + np.sum(unique_w), 4])
                else:
                    H_write = 0
                # Variance normalized Shannon Entropy (write)
                Var_H_write = (1/ (count_w - 1)) * np.sum((filtered_entropy - entropy_mean) ** 2)

                # Spatial Locality Ratio on write access
                delta = 128
                lba_values_write = filtered_write.values
                if len(lba_values_write) < 2:
                    SLR_write = 0.0
                else:
                    lba_diffs_write = np.abs(np.diff(lba_values_write))
                    SLR_write = np.sum(lba_diffs_write <= delta) / len(lba_diffs_write)

                # Spatial Locality Ratio on read access
                delta = 128
                lba_values_read = filtered_read.values
                if len(lba_values_read) < 2:
                    SLR_read = 0.0
                else:
                    lba_diffs_read = np.abs(np.diff(lba_values_read))
                    SLR_read = np.sum(lba_diffs_read <= delta) / len(lba_diffs_read)

                tmp.append([T_write, T_read, V_write, V_read, H_write, Var_H_write, SLR_write, SLR_read])

                # Move window based on read/write count
                i_r += unique_count_r[k]+1
                i_w += unique_count_w[k]+1

            tmp_train.append(tmp)
            tmp_train = np.array(tmp_train)
            transposed_array = tmp_train.transpose(1, 2, 0)

            # Combine the groups into a single array
            result_array = transposed_array.reshape(transposed_array.shape[0], -1)
            save_numpy_array(result_array, label, output_dir, file_format='csv')


AESCrypt-20200427_17-28-53


AESCrypt-20200427_16-29-10
AESCrypt-20200427_17-00-04
AESCrypt-20200427_16-23-28
AESCrypt-20200427_17-22-00
AESCrypt-20200427_16-36-30
AESCrypt-20200427_17-40-13
AESCrypt-20200427_16-52-26
AESCrypt-20200427_16-43-56
AESCrypt-20200427_17-04-45
Cerber-20200806_18-11-08
Cerber-20200805_23-25-39
Cerber-20200806_17-56-03
Cerber-20200805_23-19-47
Cerber-20200806_18-06-08
Cerber-20200806_17-38-41
Cerber-20200806_20-08-06
Cerber-20200806_17-49-48
Cerber-20200806_17-44-07
Cerber-20200806_18-01-07
Darkside-20210617_00-05-43
Darkside-20210608_19-00-19
Darkside-20210608_19-31-14
Darkside-20210608_18-45-00
Darkside-20210608_19-46-23
Darkside-20210608_19-07-58
Darkside-20210617_00-13-49
Darkside-20210608_19-23-29
Darkside-20210608_19-16-01
Darkside-20210608_19-38-53
Excel-20210702_00-22-07
Excel-20210701_23-41-10
Excel-20210702_00-05-58
Excel-20210701_23-35-36
Excel-20210702_00-16-52
Excel-20210701_23-49-20
Excel-20210702_00-27-20
Excel-20210702_00-00-36
Excel-20210701_23-54-57
Excel-20210702_00-11-

### Normalization

In [25]:
import os
import pandas as pd

def calculate_global_min_max_per_class(base_folder):
    """
    Calculate the global minimum and maximum values for the first 5 columns across all CSV files
    in each class folder inside the given base folder.
    
    Args:
        base_folder (str): Path to the base folder containing class subfolders with CSV files.
    """
    for class_folder in sorted(os.listdir(base_folder)):
        class_path = os.path.join(base_folder, class_folder)
        
        # Ensure it's a directory before processing
        if not os.path.isdir(class_path):
            continue
        
        global_min = {}
        global_max = {}
        
        for root, _, files in os.walk(class_path):
            for file in files:
                if file.endswith(".csv"):
                    file_path = os.path.join(root, file)
                    # Read CSV without headers, considering only the first 5 columns
                    df = pd.read_csv(file_path, header=None, usecols=range(8))
                    
                    # Calculate min and max for each column
                    for column in df.columns:
                        col_min = df[column].min()
                        col_max = df[column].max()
                        
                        if column not in global_min or col_min < global_min[column]:
                            global_min[column] = col_min
                        if column not in global_max or col_max > global_max[column]:
                            global_max[column] = col_max
        
        # Print results for each class folder
        print(f"Class: {class_folder}")
        print("  Global Minimum Values:", global_min)
        print("  Global Maximum Values:", global_max)
        print("----------------------------------")

# Example usage
base_folder = "/home/HardDisk/Satang/thesis_proj/New_45/10/raw_data_8"  # Replace with the path to your main folder
calculate_global_min_max_per_class(base_folder)


Class: AESCrypt
  Global Minimum Values: {0: 3807641.6, 1: 313548.8, 2: 7627867034.925859, 3: 154508024168.43106, 4: 0.000960103461790752, 5: 3.304188265809427e-05, 6: 0.65, 7: 0.7142857142857143}
  Global Maximum Values: {0: 49607321.6, 1: 55615436.8, 2: 2564835524686837.0, 3: 1843961229006897.5, 4: 0.7329253035494827, 5: 0.03648573066314812, 6: 0.9932677491999526, 7: 0.994044396318354}
----------------------------------
Class: Cerber
  Global Minimum Values: {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0, 6: 0.0, 7: 0.0}
  Global Maximum Values: {0: 17998284.8, 1: 13797478.4, 2: 3103431828817829.0, 3: 1046619239186563.6, 4: 0.665399614859678, 5: 0.13160221443471243, 6: 1.0, 7: 1.0}
----------------------------------
Class: Darkside
  Global Minimum Values: {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0, 6: 0.0, 7: 0.0}
  Global Maximum Values: {0: 29814579.2, 1: 15231488.0, 2: 496190099062513.7, 3: 1080694759653464.0, 4: 0.8429723745159272, 5: 0.14992458464043443, 6: 1.0, 7: 1.0}
-

In [26]:
import os
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np

base_dir = '/home/HardDisk/Satang/thesis_proj/New_45/10/raw_data_8'
output_dir = '/home/HardDisk/Satang/thesis_proj/New_45/10/raw_data_normalized_8'
os.makedirs(output_dir, exist_ok=True)

for class_name in os.listdir(base_dir):
    class_path = os.path.join(base_dir, class_name)
    if not os.path.isdir(class_path):
        continue

    dfs = []
    filenames = []

    # Step 1: Load all data
    for file in os.listdir(class_path):
        if file.endswith('.csv'):
            file_path = os.path.join(class_path, file)
            df = pd.read_csv(file_path, header=None)
            dfs.append(df)
            filenames.append((file, df))
    
    if not dfs:
        continue

    # Step 2: Concatenate and inspect
    all_data = pd.concat(dfs, ignore_index=True)

    # DEBUG: Check actual min/max per column
    print(f"\nClass: {class_name}")
    print("Original min values:\n", all_data.min().values)
    print("Original max values:\n", all_data.max().values)

    # Step 3: Fit scaler on the combined data
    scaler = MinMaxScaler()
    scaler.fit(all_data)

    # DEBUG: Verify scaler fit
    print("Scaler min_:\n", scaler.data_min_)
    print("Scaler max_:\n", scaler.data_max_)

    # Step 4: Normalize each file
    class_output_path = os.path.join(output_dir, class_name)
    os.makedirs(class_output_path, exist_ok=True)

    for file, df in filenames:
        norm_data = scaler.transform(df)

        # DEBUG: Log first row before and after
        print(f"File: {file}")
        print("Original first row:", df.iloc[0].values)
        print("Normalized first row:", norm_data[0])

        pd.DataFrame(norm_data).to_csv(
            os.path.join(class_output_path, file),
            index=False,
            header=False
        )

print("✅ Normalization complete.")



Class: Darkside
Original min values:
 [0. 0. 0. 0. 0. 0. 0. 0.]
Original max values:
 [2.98145792e+07 1.52314880e+07 4.96190099e+14 1.08069476e+15
 8.42972375e-01 1.49924585e-01 1.00000000e+00 1.00000000e+00]
Scaler min_:
 [0. 0. 0. 0. 0. 0. 0. 0.]
Scaler max_:
 [2.98145792e+07 1.52314880e+07 4.96190099e+14 1.08069476e+15
 8.42972375e-01 1.49924585e-01 1.00000000e+00 1.00000000e+00]
File: Darkside_4.csv
Original first row: [1.57255680e+06 4.13414400e+06 1.06712233e+14 1.00956518e+13
 1.39251167e-01 2.48409113e-03 9.49709865e-01 8.36288734e-01]
Normalized first row: [0.05274456 0.27142089 0.2150632  0.00934182 0.16519066 0.01656894
 0.94970986 0.83628873]
File: Darkside_36.csv
Original first row: [3.71717120e+06 7.03083520e+06 1.38072653e+13 5.52652272e+12
 1.50413504e-02 9.92097849e-04 7.74096386e-01 8.18418993e-01]
Normalized first row: [0.12467629 0.46159871 0.02782656 0.00511386 0.01784323 0.00661731
 0.77409639 0.81841899]
File: Darkside_38.csv
Original first row: [1.37839616e+07 

File: Darkside_39.csv
Original first row: [1.29598464e+07 8.97044480e+06 8.54266126e+13 3.83283698e+14
 1.01786997e-01 1.61129075e-03 9.73425323e-01 6.95878781e-01]
Normalized first row: [0.43468151 0.5889408  0.17216509 0.35466416 0.12074773 0.01074734
 0.97342532 0.69587878]
File: Darkside_22.csv
Original first row: [1.92435200e+06 5.06741760e+06 9.86144861e+13 6.79255233e+12
 1.50224849e-01 2.76472971e-03 9.17090909e-01 8.41785526e-01]
Normalized first row: [0.06454399 0.33269354 0.19874336 0.00628536 0.17820851 0.0184408
 0.91709091 0.84178553]
File: Darkside_23.csv
Original first row: [2.27174400e+06 5.02430720e+06 7.36767572e+13 6.95651369e+12
 1.22680596e-01 1.72337090e-03 7.73109244e-01 8.50487403e-01]
Normalized first row: [0.07619574 0.32986319 0.14848494 0.00643708 0.14553335 0.01149492
 0.77310924 0.8504874 ]
File: Darkside_29.csv
Original first row: [1.35910400e+06 4.72529920e+06 1.37932918e+14 6.61079756e+12
 2.10648967e-01 3.20980910e-03 7.79259259e-01 8.61059766e-01]
No